In [1]:
%load_ext autoreload
%autoreload 2

In [2]:
import sys
import os

PROJECT_ROOT = os.path.abspath("..")
if PROJECT_ROOT not in sys.path:
    sys.path.insert(0, PROJECT_ROOT)


In [3]:
import pyspark
from pyspark.sql import SparkSession
from pyspark.sql import functions as F

In [4]:
spark= SparkSession.builder.appName("Quality").getOrCreate()

Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
25/12/20 23:00:58 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable


In [5]:
from pyspark.sql import Row

data = [
    # ✅ valid row
    Row(song_id=1, artist_id=10, artist_name="Artist A", song_title="Song X", duration=210, year=2020),

    # ❌ duplicate PK (song_id = 1)
    Row(song_id=1, artist_id=10, artist_name="Artist A", song_title="Song X", duration=210, year=2020),

    # ❌ NULL artist_name
    Row(song_id=2, artist_id=11, artist_name=None, song_title="Song Y", duration=180, year=2019),

    # ❌ duration <= 0
    Row(song_id=3, artist_id=12, artist_name="Artist C", song_title="Song Z", duration=0, year=2018),

    # ❌ duplicate (artist_id, song_title)
    Row(song_id=4, artist_id=13, artist_name="Artist D", song_title="Song W", duration=200, year=2021),
    Row(song_id=5, artist_id=13, artist_name="Artist D", song_title="Song W", duration=200, year=2021),

    # ❌ NULL song_title
    Row(song_id=6, artist_id=14, artist_name="Artist E", song_title=None, duration=190, year=2022),
]

df_test = spark.createDataFrame(data)
df_test.show(truncate=False)
df_test.printSchema()

df_test.createOrReplaceTempView("songs_raw")


+-------+---------+-----------+----------+--------+----+
|song_id|artist_id|artist_name|song_title|duration|year|
+-------+---------+-----------+----------+--------+----+
|1      |10       |Artist A   |Song X    |210     |2020|
|1      |10       |Artist A   |Song X    |210     |2020|
|2      |11       |NULL       |Song Y    |180     |2019|
|3      |12       |Artist C   |Song Z    |0       |2018|
|4      |13       |Artist D   |Song W    |200     |2021|
|5      |13       |Artist D   |Song W    |200     |2021|
|6      |14       |Artist E   |NULL      |190     |2022|
+-------+---------+-----------+----------+--------+----+

root
 |-- song_id: long (nullable = true)
 |-- artist_id: long (nullable = true)
 |-- artist_name: string (nullable = true)
 |-- song_title: string (nullable = true)
 |-- duration: long (nullable = true)
 |-- year: long (nullable = true)



In [6]:
# dp/core/lineage.py
def extract_sources(df):
    try:
        return list(
            set(
                node.identifier
                for node in df._jdf.queryExecution().analyzed().collectLeaves()
                if hasattr(node, "identifier")
            )
        )
    except Exception:
        return []


In [7]:
extract_sources(df=df_test)

[]

In [9]:
from quality import pipelines as dp

@dp.materialized_view(
    comment="Test songs dataset",
    fail_fast=False  # important so all checks run
)

@dp.expect_no_duplicates(
    name="check duplicates",
    columns=["artist_name", "song_title"],
    severity="WARN"
)
@dp.expect_primary_key(
    name="valid_primary_key",
    columns="song_id",
    severity="ERROR"
)
@dp.expect(name="valid_year", rule="year BETWEEN 2018 AND 2020", severity="ERROR")
@dp.expect(
    name="valid_duration",
    rule="duration > 0",
    severity="ERROR"
)
def songs_test():
    return spark.read.table("songs_raw")


In [10]:
try:
    result_df = songs_test()
    result_df.show()
except Exception as e:
    print("Pipeline failed:", e)


+-------+---------+-----------+----------+--------+----+
|song_id|artist_id|artist_name|song_title|duration|year|
+-------+---------+-----------+----------+--------+----+
|      1|       10|   Artist A|    Song X|     210|2020|
|      1|       10|   Artist A|    Song X|     210|2020|
|      2|       11|       NULL|    Song Y|     180|2019|
|      3|       12|   Artist C|    Song Z|       0|2018|
|      4|       13|   Artist D|    Song W|     200|2021|
|      5|       13|   Artist D|    Song W|     200|2021|
|      6|       14|   Artist E|      NULL|     190|2022|
+-------+---------+-----------+----------+--------+----+

